In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from agent.components.GaussianProcess import GASK
from agent.components.commons import ServiceType

df = pd.read_csv("../statics/metrics_20_0.csv")
# 2. Initialize and train
rask_gp = GASK(show_figures=False)
rask_gp.init_models(df, density=1.0)

INFO:GP_Model:Fitting GP for elastic-workbench-qr-detector - Target: max_tp
INFO:multiscale:train_gp_models took 687 ms to execute


In [2]:
import numpy as np
%load_ext autoreload
%autoreload 2

rask_gp.predict(ServiceType.QR, "max_tp", {'data_quality': 100, 'cores': 6.0})

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


(np.float64(2050.112908336468), np.float64(145.8892622660747))

In [3]:
from agent.components.GaussianProcess import get_empirical_boundaries
%load_ext autoreload
%autoreload 2

from agent.components.Optimizer import local_obj, solve_global
from agent.components.SLORegistry_v2 import SLO_Registry

slo_lib = SLO_Registry("../statics/config/service_level_objectives.yml")
slos = slo_lib.get_slo_for_client("experiment-1", "client-1")

# empirical_bounds = get_empirical_boundaries(rask_gp)[ServiceType.QR]
# print(empirical_bounds)
# x = normalize_value_in_bounds({'cores': 1.0, 'data_quality': 990}, empirical_bounds)
x = [2.0, 400]

# local_obj(x, slos, rask_gp)
solve_global(slos, rask_gp, last_assignments=x)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


[np.float64(6.0), np.float64(990.0)]

In [4]:
import numpy as np

# Convert to a numpy array so we can do math on the whole vector
x_norm = np.array([0.1, 0.1])
raw_bounds = get_empirical_boundaries(rask_gp.training_data)[ServiceType.QR]
del raw_bounds['max_tp']

# Store these to use for de-normalization inside the objective
ordered_bounds = list(raw_bounds.values())

for e in [1e-5, 1e-3, 1e-2, 5e-2]:
    # val_start uses the original center
    val_start = local_obj(x_norm, slos, rask_gp, ordered_bounds)

    # x_norm + e now adds 'e' to every element (e.g., [0.11, 0.11])
    val_nudge = local_obj(x_norm + e, slos, rask_gp, ordered_bounds)

    diff = abs(val_start - val_nudge)
    print(f"Eps {e}: Change in SLO-F is {diff:.6f}")

Eps 1e-05: Change in SLO-F is 0.000008
Eps 0.001: Change in SLO-F is 0.000763
Eps 0.01: Change in SLO-F is 0.007398
Eps 0.05: Change in SLO-F is 0.029776


In [6]:
from agent.components.Optimizer import VersatileMapElites

# 1. Initialize
v_me = VersatileMapElites(bins=10)

# 2. Run the illumination
v_me.run_search(slos, rask_gp, ordered_bounds, iterations=1000)

# 3. Get 5 solutions that are high-performing but far apart
diverse_set = v_me.get_diverse_set(n_solutions=10, versatility=0.2)

# 4. De-normalize for your agent
for sol in diverse_set:
    # real_val = [sol[i] * (ordered_bounds[i][1] - ordered_bounds[i][0]) + ordered_bounds[i][0]
    #             for i in range(2)]
    print(f"Versatile Candidate: {sol}")

Iteration 0: Discovered new Elite in bin (np.int64(8), np.int64(5)) with fitness 0.6384
Iteration 100: Discovered new Elite in bin (np.int64(0), np.int64(8)) with fitness 0.2416
Iteration 200: Discovered new Elite in bin (np.int64(8), np.int64(0)) with fitness 0.8056
Iteration 600: Discovered new Elite in bin (np.int64(2), np.int64(9)) with fitness 0.3703
Versatile Candidate: {'coord': array([1.        , 0.12789294]), 'fitness': np.float64(0.8092218990353979)}
Versatile Candidate: {'coord': array([0.7759727 , 0.13412495]), 'fitness': np.float64(0.7625041778058422)}
Versatile Candidate: {'coord': array([1., 1.]), 'fitness': np.float64(0.7)}
Versatile Candidate: {'coord': array([1.        , 0.33630218]), 'fitness': np.float64(0.6707668735002895)}
Versatile Candidate: {'coord': array([0.93577313, 0.76310211]), 'fitness': np.float64(0.6691115514066628)}
Versatile Candidate: {'coord': array([0.77378004, 0.90353894]), 'fitness': np.float64(0.6482289758879268)}
Versatile Candidate: {'coord': 